[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/03_document_input/03_document_input.ipynb)

# 03. `document-input-example` 동행 노트북

> 대상 프로젝트: [`example-projects/document-input-example`](../../../example-projects/document-input-example) (B파트: 사용자 입력)
> · 다른 선택지: [ALTERNATIVES.md](../../../example-projects/document-input-example/ALTERNATIVES.md)

## 이 장을 배우는 이유

LLM에게 "이 서류 내용 정리해줘"라고 하면 답을 잘 해줍니다. 그런데 그 답을 **프로그램이 받아서
써야 한다면** 문제가 생깁니다. 매번 형식이 다르기 때문입니다.

```
1회차: "이 서류는 휴가신청서이고, 신청자는 3월 2일부터 6일까지 휴가를 원합니다."
2회차: "문서 종류: 휴가신청서 / 기간: 2026.3.2~3.6"
3회차: "휴가신청서입니다. 기간은 3월 초순이며..."
```

사람은 다 읽을 수 있지만, **코드로 "기간"만 꺼내려면 매번 다른 파싱이 필요합니다.**
게다가 서류에 안 적힌 내용을 LLM이 지어내도 알아챌 방법이 없습니다.

그래서 이 프로젝트는 LLM에게 자유롭게 답하게 두지 않고,
**빈칸이 정해진 양식을 주고 거기에만 채워 넣게 강제합니다.** 그게 [정형 출력](../../../glossary.md#structured-output)입니다.

```
01 crawl-storage → 02 preprocess → [03 document-input] → 04 rag-regulation
                                         여기
```

이번 장에서 배우는 것

- `Field(description=...)`이 주석이 아니라 **LLM이 실제로 읽는 지시문**이라는 것
- "모르면 비워두라"고 명시하는 것이 [환각](../../../glossary.md#hallucination)을 줄이는 이유
- Pydantic 검증이 **문제를 가장 이른 시점에 터뜨리는** 안전망이 되는 순간 (직접 깨뜨려봅니다)
- `schema.py`와 `structurer.py`를 나눈 이유
- 설정 검증을 시작 시점에 하는 것(fail fast)과 안 했을 때의 차이
- Streamlit에서 버튼 뒤에 API 호출을 두는 이유 (비용)

**소요 시간**: 30~40분. **Google Vision 인증도 API 키도 없이 끝까지 실행됩니다.**
[OCR](../../../glossary.md#ocr)은 결과를 흉내 낸 텍스트로, LLM 호출은 규칙 기반 함수로 대체합니다.
(`OPENAI_API_KEY`가 있으면 그 부분만 실제로 호출합니다.)

## 이 노트북을 읽는 법

- **셀을 위에서부터 순서대로 실행하세요**(`Shift + Enter`).
- **실행 결과는 저장되어 있지 않습니다.** 직접 실행해야 출력이 나타납니다.
- 코드 셀 앞에는 **지금 무엇을 할 것인지**를 적어두었습니다. 셀 뒤에는 두 가지가 붙습니다 —
  `show()`로 프로젝트 소스를 펼친 뒤에는 **코드에서 짚을 곳**이, 실제로 돌려본 뒤에는
  **결과 읽는 법**이 나옵니다.
- 앞의 01·02를 안 봐도 따라갈 수 있습니다. 이 프로젝트는 별개의 입력 경로입니다.
- 낯선 용어는 [glossary.md](../../../glossary.md)에서 찾아보세요.
- **에러가 나거나 결과가 예상과 다르면** [troubleshooting.md](../../../troubleshooting.md)를 먼저 보세요.
  설치 실패, 한글 깨짐, `NameError`, API 키, GPU 설정처럼 여러 노트북에서 반복되는 문제를 모아뒀습니다.

## 이 프로젝트의 자리 — 앞의 둘과 방향이 다릅니다

01·02는 **미리 긁어둔 규정 문서**를 검색 가능하게 만드는 경로였습니다.
이 프로젝트는 그 줄기에 이어 붙는 게 아니라, **별개의 입력 경로**입니다.

```
[A 경로] 웹 크롤링 -> 원본 보관 -> 전처리 -> 색인      (미리 쌓아두는 쪽)
[B 경로] 사용자가 방금 올린 서류 사진 -> OCR -> 정형 JSON   ← 이 프로젝트
                                          |
                     JSON의 키워드가 C파트(RAG)의 "질문"이 된다
```

시나리오는 이렇습니다. 직원이 휴가신청서를 사진으로 찍어 올립니다.
시스템이 그걸 읽고 "아, 이건 육아휴직 신청서고, 2026년 3월부터 쓰겠다는 거구나"를 파악한 뒤,
**그 내용으로 규정을 자동 검색해서** "육아휴직은 최대 12개월입니다" 같은 안내를 붙여줍니다.

이 프로젝트는 그중 **"사진 → 구조화된 데이터"** 부분을 담당합니다.

## 막혔을 때 — 이 노트북에서 자주 나오는 증상

| 증상 | 원인 | 해볼 것 |
|---|---|---|
| `AssertionError: 프로젝트 경로를 찾지 못했습니다` | **이 노트북은 실제 프로젝트 파일을 열어야 돌아갑니다.** Colab이라면 `git clone`이 실패했고, 로컬이라면 저장소 밖에서 노트북을 열었습니다 | Colab: 네트워크·프록시 확인 후 첫 셀 재실행. 로컬: 저장소를 통째로 받아 원래 폴더 구조 그대로 열기 |
| `show(...)`에서 `FileNotFoundError` | 파일명 오타이거나 프로젝트 구조가 바뀜 | `import os; print(os.listdir(SRC))`로 실제 파일 목록 확인 |
| import한 함수가 예전 동작을 한다 | 프로젝트 파일을 수정했지만 파이썬이 이미 불러둔 모듈을 재사용 | `import importlib; importlib.reload(모듈)` 또는 런타임 재시작 |
| `ModuleNotFoundError` — 프로젝트 모듈을 못 찾음 | `sys.path.insert(0, SRC)` 셀을 건너뜀 | 맨 위 준비 셀부터 순서대로 실행 |
| 결과가 투박하다 / `HAS_REAL_KEY`가 `False` | **API 키 없이 돌도록 규칙 기반 함수로 대체**됩니다 | 정상이고 의도된 동작입니다 |
| 진짜 키를 넣었는데도 규칙 기반으로 돈다 | 준비 셀이 `os.environ.setdefault`로 **더미 키를 이미 심어놨습니다** | `os.environ["OPENAI_API_KEY"] = "진짜 키"`로 **덮어쓴 뒤** 아래 셀을 다시 실행 |
| Google Vision 인증 에러 | 준비 셀이 만드는 인증 파일은 **가짜(placeholder)** 입니다 | OCR은 흉내 낸 텍스트로 대체합니다. 실제 호출 셀은 없습니다 |
| Streamlit 화면이 안 뜬다 | `app.py`는 `show()`로 **읽기만** 하고 노트북에서 띄우지 않습니다 | 실제로 띄우려면 프로젝트 폴더에서 `streamlit run src/app.py` |

여기 없는 문제(설치 실패, 한글 깨짐, API 키 설정 방법)는 [troubleshooting.md](../../../troubleshooting.md)에 모아뒀습니다.

## 0. 환경 준비 — 프로젝트를 옆에 펼쳐두기

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab에서 실행 중:", IN_COLAB)

if IN_COLAB:
    # 이 노트북은 "예제 프로젝트를 옆에 두고 같이 읽는" 노트북입니다.
    # 그래서 설명만 하지 않고, 저장소를 통째로 내려받아 **실제 프로젝트 파일**을 열어봅니다.
    subprocess.run(["git", "clone", "-q", "https://github.com/karzit/temp.git", "/content/temp"], check=False)
    REPO_ROOT = "/content/temp"
    !pip install -q pydantic openai
else:
    # 로컬에서 열었다면 이 노트북 위치(notebooks/project-walkthrough/NN_xxx/)에서 3단계 위가 저장소 루트입니다.
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))

PROJECT = os.path.join(REPO_ROOT, "example-projects", "document-input-example")
SRC = os.path.join(PROJECT, "src")
print("프로젝트 경로:", PROJECT)
assert os.path.isdir(SRC), "프로젝트 경로를 찾지 못했습니다. 저장소 루트에서 노트북을 열었는지 확인하세요."

아래 `show()`는 이 노트북 전체에서 쓰는 도우미입니다. **설명 대신 진짜 프로젝트 파일을 그대로 출력**해서, 노트북과 코드가 어긋나지 않게 합니다.

In [ ]:
import re


def show(filename, start=None, end=None, grep=None):
    """프로젝트 파일의 실제 소스를 줄 번호와 함께 출력한다.

    설명을 읽는 것과 실제 코드를 보는 것 사이의 간격을 없애기 위한 도우미입니다.
    이 노트북에서 "코드 읽기"라고 나오는 곳은 전부 진짜 프로젝트 파일을 그대로 보여줍니다.

        show("crawl.py")                  전체
        show("crawl.py", 30, 45)          30~45번째 줄
        show("crawl.py", grep="def ")     'def '가 들어간 줄만
    """
    path = os.path.join(SRC, filename) if not os.path.isabs(filename) else filename
    lines = open(path, encoding="utf-8").read().splitlines()

    if grep:
        picked = [(i, l) for i, l in enumerate(lines, 1) if re.search(grep, l)]
    else:
        s = (start or 1) - 1
        e = end or len(lines)
        picked = [(i, l) for i, l in enumerate(lines[s:e], s + 1)]

    for i, line in picked:
        print(f"{i:>4} | {line}")


def show_file(relpath, **kwargs):
    """프로젝트 루트 기준 경로로 파일을 보여준다 (README, docker-compose 등)."""
    show(os.path.join(PROJECT, relpath), **kwargs)


# 프로젝트 소스를 import할 수 있도록 경로를 등록해둡니다.
if SRC not in sys.path:
    sys.path.insert(0, SRC)

`config.py`는 Google Vision 인증 파일이 없으면 **즉시 에러를 냅니다.**
그 설계 자체가 이 프로젝트에서 배울 점 중 하나라 나중에 따로 보고,
지금은 노트북이 돌아가도록 가짜 인증 파일을 만들어둡니다.

In [ ]:
import json

os.environ.setdefault("OPENAI_API_KEY", "sk-dummy-replace-if-you-have-one")

# config.py의 검증을 통과하기 위한 가짜 인증 파일 (실제 호출에는 쓸 수 없습니다)
fake_key_path = os.path.abspath("fake-google-key.json")
with open(fake_key_path, "w", encoding="utf-8") as f:
    json.dump({"type": "service_account", "note": "notebook placeholder"}, f)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = fake_key_path

for name in sorted(os.listdir(SRC)):
    path = os.path.join(SRC, name)
    # import를 한 번이라도 하면 SRC 안에 __pycache__ 디렉터리가 생긴다.
    # 아래에서 open()으로 줄 수를 세므로, 파일이 아닌 것은 먼저 걸러낸다.
    if not os.path.isfile(path):
        continue
    print(f"  {name:<16} {len(open(path, encoding='utf-8').read().splitlines()):>3}줄")

| 파일 | 역할 |
|---|---|
| `schema.py` | **무엇을** 뽑을지 — 데이터의 양식 |
| `structurer.py` | **어떻게** 채울지 — LLM 호출 |
| `ocr.py` | 사진에서 글자 읽기 |
| `app.py` | Streamlit 화면 |
| `config.py` | 설정 + 시작 시점 검증 |

`schema.py`와 `structurer.py`를 나눈 게 이 프로젝트의 핵심 설계입니다. 이유는 4번에서 봅니다.

## 1. `schema.py` — LLM에게 줄 "빈칸 있는 양식"

읽는 순서는 여기부터입니다. **데이터의 모양을 먼저 알아야 나머지가 이해됩니다.**

In [ ]:
show("schema.py", grep="class |    document_type|    applicant_request|    related_dates|    keywords|description=")

**코드에서 짚을 곳** — Pydantic `BaseModel`을 상속하면 각 필드의 **타입**이 정해집니다.
그런데 여기서 더 중요한 건 `Field(description=...)`입니다.

**이 설명문은 주석이 아닙니다. LLM이 실제로 읽는 지시문입니다.**

직접 확인해봅시다. Pydantic 모델을 JSON 스키마로 변환하면, OpenAI에 전달되는 형태가 나옵니다.

In [ ]:
from schema import RegulationInquiry

print(json.dumps(RegulationInquiry.model_json_schema(), ensure_ascii=False, indent=2))

**결과 읽는 법**

`description`이 스키마 안에 그대로 들어가 있는 게 보이시나요?
OpenAI는 이 스키마를 받아서 **"이 틀에 맞춰서만 답해라"**는 제약으로 씁니다.

그래서 `description`을 잘 쓰는 게 프롬프트를 잘 쓰는 것과 같습니다.

| 나쁜 예 | 좋은 예 |
|---|---|
| `Field(description="날짜")` | `Field(description="서류에 등장하는 날짜들 (YYYY-MM-DD 형식으로, 없으면 빈 목록)")` |

형식(`YYYY-MM-DD`)과 예외 상황(`없으면 빈 목록`)까지 적어주면 결과가 훨씬 안정적입니다.
실제 프로젝트의 `related_dates` 필드가 정확히 그렇게 되어 있죠.

## 2. `ocr.py` — OCR 결과는 지저분하다는 전제

Google Vision 호출은 인증이 필요해서 실행할 수 없지만, **코드는 짧고 읽을 만합니다.**

In [ ]:
show("ocr.py", grep="def extract_text_from_image|ImageAnnotatorClient|vision.Image|document_text_detection|error.message|full_text_annotation")

**코드에서 짚을 곳** — 두 군데입니다.

**① `document_text_detection`을 쓴 것** — Vision API에는 `text_detection`도 있습니다.
간판 글씨 하나 읽는 것과, 문서 전체의 문단 구조를 읽는 것은 다른 기능입니다.
서류를 다루니 후자를 골랐습니다.

**② 에러를 즉시 예외로 바꾼 것** — Google SDK는 실패해도 예외를 던지지 않고
`response.error.message`에 담아서 돌려줍니다. 이걸 확인하지 않으면
**빈 문자열이 정상 결과인 것처럼 조용히 흘러갑니다.** 조용한 실패가 제일 무섭습니다.

그리고 파일 맨 위 docstring에 이렇게 적혀 있습니다 — **"OCR 결과는 완벽하지 않다."**
이 전제가 다음 단계의 존재 이유입니다. OCR 원문을 흉내 내보면 왜 그런지 바로 보입니다.

In [ ]:
# 실제 OCR 결과는 이런 식입니다. 줄바꿈이 뒤죽박죽이고 오타가 섞입니다.
ocr_text = """휴 가 신 청 서

성명: 김민수      부서: 개발l팀
신청구분: [v] 연차휴가  [ ] 경조휴가  [ ] 병가

기간: 2026 . 3 . 2 ~ 2026 . 3 . 6  (5일간)

사유: 가족 여행으로 인한 연차 사용입니다.
      3월 2일부터 6일까지 자리를 비웁니다.

신청일: 2026-02-20
"""
print(ocr_text)

**결과 읽는 법**

`개발l팀`(숫자 1이 아니라 알파벳 l), `2026 . 3 . 2`처럼 공백이 낀 날짜, 체크박스 기호…
이걸 **그대로 검색에 쓸 수는 없습니다.** 그래서 AI에게 한 번 정리를 시킵니다.

## 3. `structurer.py` — 양식에 채워 넣게 만들기

In [ ]:
show("structurer.py", grep="SYSTEM_PROMPT|OCR로 인식된|텍스트에 없는|def structure_text|client = |parse\\(|model=|response_format|return completion")

**코드에서 짚을 곳**

`SYSTEM_PROMPT`에서 가장 중요한 문장은 마지막 줄입니다.

> **"텍스트에 없는 내용은 추측해서 지어내지 말고, 알 수 없으면 빈 값으로 두세요."**

LLM은 빈칸을 보면 채우고 싶어 합니다. 그게 **환각(hallucination)** 입니다.
서류에 부서가 안 적혀 있는데 "개발팀"이라고 지어내면, 그 값이 그대로 시스템에 흘러갑니다.
**"모르면 비워두라"고 명시적으로 허락**해주는 것이 환각을 줄이는 가장 값싼 방법입니다.

`beta.chat.completions.parse()`에 `response_format=RegulationInquiry`를 넘기면,
응답이 자동으로 그 Pydantic 객체로 변환되어 돌아옵니다.
JSON 문자열을 직접 파싱하고 검증하는 과정이 통째로 사라집니다.

API 키가 있으면 실제로, 없으면 규칙 기반 대체 함수로 돌려보겠습니다.

In [ ]:
import re

from schema import RegulationInquiry

HAS_REAL_KEY = os.environ.get("OPENAI_API_KEY", "").startswith("sk-") and "dummy" not in os.environ.get(
    "OPENAI_API_KEY", ""
)


def structure_text_fallback(raw_text: str) -> RegulationInquiry:
    """API 키가 없을 때 쓰는 규칙 기반 대체. LLM이 하는 일을 정규식으로 흉내만 냅니다."""
    dates = re.findall(r"\d{4}\s*[-.]\s*\d{1,2}\s*[-.]\s*\d{1,2}", raw_text)
    normalized = ["-".join(f"{int(p):02d}" if i else p for i, p in enumerate(re.split(r"[-.]", d.replace(" ", ""))))
                  for d in dates]
    return RegulationInquiry(
        document_type="휴가신청서" if "휴가" in raw_text else "미상",
        applicant_request="연차휴가 5일 사용 신청",
        related_dates=normalized,
        keywords=[w for w in ("연차휴가", "경조휴가", "병가") if w in raw_text],
    )


if HAS_REAL_KEY:
    from structurer import structure_text

    result = structure_text(ocr_text)
    print("(실제 OpenAI 호출)")
else:
    result = structure_text_fallback(ocr_text)
    print("(API 키가 없어 규칙 기반 대체 함수를 사용했습니다)")

print(result.model_dump_json(indent=2))

**결과 읽는 법** — 지저분한 OCR 원문이 **항목별로 값이 정해진 데이터**가 됐습니다.
이제 `result.keywords`를 그대로 규정 검색의 질문으로 넘길 수 있습니다.

## 4. 스키마가 안전망이 되는 순간 — 직접 깨뜨려보기

`schema.py`와 `structurer.py`를 나눈 이유가 여기서 드러납니다.
LLM이 이상한 값을 돌려주면 어떻게 될까요?

In [ ]:
from pydantic import ValidationError

bad_outputs = [
    {"document_type": "휴가신청서", "applicant_request": "연차 신청", "related_dates": "2026-03-02", "keywords": []},
    {"document_type": "휴가신청서", "keywords": ["연차휴가"]},
    {"document_type": 12345, "applicant_request": "연차 신청"},
]

for i, bad in enumerate(bad_outputs, 1):
    print(f"--- 케이스 {i}: {bad}")
    try:
        RegulationInquiry(**bad)
        print("    통과")
    except ValidationError as e:
        for err in e.errors():
            print(f"    ✗ {'.'.join(str(x) for x in err['loc'])}: {err['msg']}")
    print()

**결과 읽는 법**

- **케이스 1** — `related_dates`에 리스트가 아니라 문자열 하나가 왔습니다. 잡힙니다.
- **케이스 2** — `applicant_request`가 통째로 빠졌습니다. 잡힙니다.
- **케이스 3** — `document_type`에 숫자가 왔습니다. 잡힙니다.

**이게 없으면 어떻게 될까요?** 잘못된 값이 조용히 통과해서 DB에 저장되고,
한참 뒤 전혀 다른 곳에서 `TypeError`가 터집니다. 그때는 원인을 찾기가 훨씬 어렵습니다.

스키마는 **문제를 가장 이른 시점에 터뜨리는 장치**입니다.
그리고 파일이 나뉘어 있으니, 필드를 추가할 때 `schema.py`만 보면 됩니다.
LLM 호출 방식(`structurer.py`)과 데이터 모양(`schema.py`)은 서로 다른 이유로 바뀌니까요.

## 5. `config.py` — 실패를 언제 알아채게 할 것인가

같은 철학이 설정 검증에도 적용돼 있습니다.

In [ ]:
show("config.py", grep="OPENAI_API_KEY|_GOOGLE_CREDENTIALS_PATH|raise RuntimeError|if not|os.path.isfile")

**코드에서 짚을 곳** — 이 검증이 **없다면** 이런 일이 벌어집니다.

```
앱 실행           → 정상 (아무 에러 없음)
사용자가 사진 업로드 → 정상
"분석 시작" 클릭    → 알아보기 힘든 구글 SDK 인증 에러
```

사용자가 다 하고 나서야 실패합니다. 그것도 개발자만 알아볼 수 있는 메시지로요.

검증을 넣어두면 **앱을 켜는 순간** 친절한 한글 메시지와 함께 멈춥니다.
심지어 발급 방법 링크까지 적혀 있죠. 이걸 **fail fast**라고 합니다.

> 💡 `OPENAI_API_KEY`는 `os.environ["..."]`로 읽는 것만으로 같은 효과를 냅니다.
> 없으면 `KeyError`가 즉시 나니까요. `GOOGLE_APPLICATION_CREDENTIALS`는 우리가 직접
> 읽지 않고 SDK가 알아서 찾아 쓰는 값이라, **일부러** 검증 코드를 넣어준 겁니다.

## 6. `app.py` — 화면과 비용

Streamlit 부분은 짧습니다. 하지만 설계 판단이 하나 숨어 있습니다.

In [ ]:
show("app.py", grep="file_uploader|st.button|st.spinner|st.image|st.subheader|st.json|model_dump_json")

**코드에서 짚을 곳**

**`if st.button("분석 시작"):` 안에 호출을 둔 이유**를 보세요.

Streamlit은 사용자가 뭔가 할 때마다 **스크립트 전체를 처음부터 다시 실행**합니다.
버튼 없이 업로드만으로 분석이 돌아가게 짜면, 사용자가 이미지를 세 번 바꿀 때마다
OCR과 LLM이 세 번 호출됩니다. **둘 다 호출당 과금됩니다.**

버튼 하나가 비용을 사용자 의도에 묶어두는 장치인 셈입니다.

`st.spinner`로 진행 상황을 보여주는 것도 중요합니다. OCR과 LLM 호출은 각각 몇 초씩 걸리는데,
아무 표시가 없으면 사용자는 고장 났다고 생각하고 새로고침합니다. (그리고 다시 과금됩니다.)

## 7. 이 JSON이 C파트로 어떻게 넘어가나

마지막으로 이 프로젝트의 출력이 어디로 가는지 확인합니다.

In [ ]:
print("추출된 키워드:", result.keywords)
print("요청 내용    :", result.applicant_request)
print()
print("이 값들이 C파트(rag-regulation-example)의 질문이 됩니다:")
print()
question = f"{result.applicant_request} 관련 규정을 알려주세요. 키워드: {', '.join(result.keywords)}"
print(f'  query.answer("{question}")')

**결과 읽는 법** — 사용자는 질문을 **한 글자도 타이핑하지 않았습니다.** 사진 한 장 올렸을 뿐이죠.
서류를 읽어서 자동으로 질문을 만들어내는 것 — 이게 B파트가 존재하는 이유입니다.

실제 검색과 답변 생성은 [`04_rag_regulation`](../04_rag_regulation/04_rag_regulation.ipynb)에서 다룹니다.

## 정리

```
서류 사진 -> ocr.py (Vision) -> 지저분한 텍스트
          -> structurer.py (LLM + 스키마) -> 검증된 JSON
          -> C파트의 질문으로
```

| 결정 | 이유 |
|---|---|
| `schema.py`와 `structurer.py` 분리 | 데이터 모양과 호출 방식이 서로 다른 이유로 바뀜 |
| `Field(description=...)`을 꼼꼼히 | 주석이 아니라 **LLM이 읽는 지시문** |
| "모르면 비워두라"고 명시 | 환각을 줄이는 가장 값싼 방법 |
| Pydantic 검증 | 문제를 가장 이른 시점에 터뜨림 |
| `config.py`의 시작 시점 검증 | fail fast — 사용자가 다 하고 나서 실패하지 않게 |
| 버튼 뒤에 API 호출 | 과금을 사용자 의도에 묶어둠 |
| `document_text_detection` 선택 | 간판 글씨용이 아니라 문서용 기능 |

**가장 기억할 것**: **LLM의 출력을 믿지 말고 검증하세요.**
스키마는 LLM을 의심하는 장치입니다. 잘 동작할 때는 있으나 마나지만,
이상한 값이 나온 그 한 번에 시스템 전체를 지켜줍니다.

**스스로 확인해보기**

- [ ] `Field(description=...)`이 LLM에게 어떻게 전달되는지 `model_json_schema()`로 확인했다
- [ ] "모르면 비워두라"를 프롬프트에 넣는 이유를 설명할 수 있다
- [ ] Pydantic 검증이 없으면 잘못된 값이 어디까지 흘러가는지 안다
- [ ] `schema.py`와 `structurer.py`를 나눈 이유를 말할 수 있다
- [ ] Streamlit에서 버튼 없이 짜면 무슨 일이 생기는지 안다

## 연습 문제

**1. 필드 추가하기**
`RegulationInquiry`에 `applicant_name`(신청자 이름)과 `department`(부서)를 추가해보세요.
`description`을 어떻게 써야 OCR 오타(`개발l팀`)가 있어도 잘 채워질까요?

**2. 신뢰도 표시하기**
OCR이 흐릿해서 확신이 없을 때 이를 알 방법이 없습니다.
`confidence: float` 같은 필드를 추가한다면 `description`에 뭐라고 써야 할까요?
그리고 그 값이 낮을 때 앱은 어떻게 동작해야 할까요?

**3. 검증 규칙 강화하기**
`related_dates`에 `"2026-13-45"` 같은 불가능한 날짜가 들어와도 지금은 통과합니다
(타입만 `str`이니까요). Pydantic의 `field_validator`로 실제 날짜인지 검사해보세요.
**LLM을 믿지 않고 한 번 더 검사하는 게 왜 필요한지** 함께 생각해보세요.

**해설/정답**: [03_document_input_solutions.ipynb](03_document_input_solutions.ipynb)

## 다음 단계

- [`04_rag_regulation`](../04_rag_regulation/04_rag_regulation.ipynb) — 여기서 만든 질문으로 실제 규정을 검색하고 답변 생성
- 라이브러리 자체 실습: [`rag-pipeline-practice/03_document_structuring`](../../rag-pipeline-practice/03_document_structuring/03_document_structuring.ipynb)